In [2]:
import os
import time
import json
import pandas as pd
import numpy as np
import tiktoken
import wrds
import h5py
import scipy.stats as stats
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

# --- OpenAI Configuration ---
client = OpenAI()

# Models
GEN_MODEL = "gpt-4o-mini"          # For generating the 280-char summaries
EMBED_MODEL = "text-embedding-3-small" # For vectorizing the summaries
MAX_TOKENS = 8000

# --- Caching Configuration ---
CACHE_DIR = '_cache/llm_summaries'
os.makedirs(CACHE_DIR, exist_ok=True)
CACHE_FILE = os.path.join(CACHE_DIR, 'firm_summaries.json')

# --- Tensor Path ---
H5_PATH = r"C:\Users\jonat\Lasso_paper\Results\Estimation\Cross_Sectional\betas.h5"

In [3]:
print("Loading tensor to extract PERMNO universe...")
with h5py.File(H5_PATH, 'r') as f:
    tensor_permnos = f['stocks'][:].astype(str)
print(f"Found {len(tensor_permnos)} PERMNOs in the model tensor.")

def fetch_firm_identities():
    """Pulls the CRSP stocknames table to map PERMNO to Ticker and Company Name."""
    print("Establishing connection to WRDS...")
    db = wrds.Connection() 
    
    print("Querying CRSP stocknames...")
    sql_query = """
        SELECT 
            permno, 
            ticker, 
            comnam,
            nameenddt
        FROM 
            crsp.stocknames
        WHERE 
            ticker IS NOT NULL
    """
    df = db.raw_sql(sql_query)
    db.close()
    
    # Format and get the most recent ticker/name for each firm
    df.columns = [col.upper() for col in df.columns]
    df['PERMNO'] = df['PERMNO'].fillna(0).astype(int).astype(str)
    
    # Sort by date and take the last valid name/ticker for each PERMNO
    df = df.sort_values('NAMEENDDT').groupby('PERMNO').last().reset_index()
    return df

# Fetch the mapping and filter to just our tensor universe
names_df = fetch_firm_identities()
firm_df = names_df[names_df['PERMNO'].isin(tensor_permnos)].copy()

print(f"Successfully mapped {len(firm_df)} firms from the tensor.")

Loading tensor to extract PERMNO universe...
Found 1296 PERMNOs in the model tensor.
Establishing connection to WRDS...
WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\jonat\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Querying CRSP stocknames...
Successfully mapped 1296 firms from the tensor.


In [5]:
import os
import json
import concurrent.futures
from tqdm.auto import tqdm

def load_cached_summaries():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_summaries(summary_dict):
    with open(CACHE_FILE, 'w') as f:
        json.dump(summary_dict, f, indent=4)

# Notice we changed this to accept a row and return the permno along with the summary
# This is crucial so we know which firm the response belongs to when futures finish out of order!
def generate_firm_summary_parallel(row):
    permno = str(row['PERMNO'])
    ticker = str(row['TICKER'])
    comnam = str(row['COMNAM'])
    
    prompt = (
        f"Describe the core business, primary products, and industry of the company "
        f"'{comnam}' (Ticker: {ticker}). Keep it purely factual. "
        f"Maximum 280 characters."
    )
    
    try:
        response = client.chat.completions.create(
            model=GEN_MODEL,
            messages=[
                {"role": "system", "content": "You are a concise financial analyst."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=100,
            temperature=0.0 # Keep it deterministic and factual
        )
        return permno, response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error generating for {ticker}: {e}")
        return permno, None

# --- Execute LLM Generation ---
firm_summaries = load_cached_summaries()
missing_firms = firm_df[~firm_df['PERMNO'].isin(firm_summaries.keys())]

if not missing_firms.empty:
    print(f"Generating LLM summaries for {len(missing_firms)} missing firms in parallel...")
    
    MAX_WORKERS = 10  # Number of parallel requests. Lower this if you hit rate limits.
    BATCH_SIZE = 50   # How many firms to process before triggering a save.
    
    # Convert dataframe to a list of dicts/series for easy iteration
    tasks = [row for _, row in missing_firms.iterrows()]
    
    # Process in chunks to maintain our periodic saving safety net
    for i in range(0, len(tasks), BATCH_SIZE):
        batch = tasks[i:i + BATCH_SIZE]
        batch_num = (i // BATCH_SIZE) + 1
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # Submit all tasks in the current batch to the executor
            futures = {executor.submit(generate_firm_summary_parallel, row): row for row in batch}
            
            # As each parallel request finishes, process the result
            for future in tqdm(concurrent.futures.as_completed(futures), total=len(batch), desc=f"Batch {batch_num}"):
                permno, summary = future.result()
                if summary:
                    firm_summaries[permno] = summary
                    
        # Save to disk after the entire batch finishes
        save_summaries(firm_summaries)

print(f"Ready: {len(firm_summaries)} firm summaries loaded.")

Generating LLM summaries for 1296 missing firms in parallel...


Batch 1:   0%|          | 0/50 [00:00<?, ?it/s]

Batch 2:   0%|          | 0/50 [00:00<?, ?it/s]

Batch 3:   0%|          | 0/50 [00:00<?, ?it/s]

Batch 4:   0%|          | 0/50 [00:00<?, ?it/s]

...

Batch 23:   0%|          | 0/50 [00:00<?, ?it/s]

Batch 24:   0%|          | 0/50 [00:00<?, ?it/s]

Batch 25:   0%|          | 0/50 [00:00<?, ?it/s]

Batch 26:   0%|          | 0/46 [00:00<?, ?it/s]

Ready: 1296 firm summaries loaded.


In [ ]:
# --- Load Topics from Topic Model Matrix ---
# Update this to the name of your new matrix file!
TOPICS_CSV = r"C:\Users\jonat\Lasso_paper\Narratives\narratives_construction\_data\word_probabilities.csv" 
print(f"Loading topic matrix from {TOPICS_CSV}...")

# Load the CSV and set the 'term' column as the index
topic_df = pd.read_csv(TOPICS_CSV, index_col='term')

topics_dict = {}

# A small override map to ensure the keys perfectly match the betas.h5 tensor
# based on our previous validation checks.
override_map = {
    "Environment": "Environment_", # Adds the trailing underscore the tensor expects
}

# Iterate over every column (Topic) in the dataframe
for col in topic_df.columns:
    # 1. Get the top 20 terms with the highest probability/weight for this topic
    # .dropna() ensures we don't accidentally grab empty rows
    top_terms = topic_df[col].dropna().nlargest(20).index.tolist()
    
    # 2. Join them into a single string
    terms_str = " ".join(str(t).strip() for t in top_terms)
    
    # 3. Clean the column name to match the tensor format
    raw_label = str(col).strip().replace(' ', '_')
    tensor_key = override_map.get(raw_label, raw_label)
    
    # 4. Add to our dictionary
    topics_dict[tensor_key] = terms_str

# --- Add Macro Topics ---
# We append the custom 280-character macro descriptions we generated earlier
macro_topics = {
    'Fed_Funds_Effective_Rate': 'The interest rate at which depository institutions trade federal funds with each other overnight. It is the primary tool used by the Federal Reserve to guide monetary policy and influence broader economic borrowing costs.',
    '10-Year_Treasury_Yield':   'The return on investment for U.S. government debt obligations maturing in 10 years. It serves as a critical benchmark for long-term interest rates, directly influencing mortgage rates and signaling economic confidence.',
    'VIX_Volatility_Index':     'Often called the market fear gauge, the VIX measures the stock market expectation of volatility based on S&P 500 index options. High values indicate elevated uncertainty and risk expectations over the next 30 days.',
    'Financial_Stress_Index':   'A composite measure designed to evaluate the current level of systemic risk and stress in financial markets. It tracks credit spreads, volatility, and liquidity to signal potential economic disruptions.',
    'High_Yield_Option-Adjusted_Spread': 'The yield difference between risky junk bonds and risk-free Treasury bonds, adjusted for embedded options. It reflects the premium investors demand to take on corporate default risk in the high-yield credit market.',
    'Trade_Weighted_USD_Index': 'A measure of the U.S. dollar value relative to a basket of foreign currencies from major trading partners. It provides a broad view of the dollar international purchasing power and exchange rate strength.',
    '3-Month_Treasury_Yield':   'The annualized return on short-term U.S. government debt maturing in three months. It is a key indicator of money market conditions and closely tracks the Federal Reserve near-term interest rate policies.',
    'WTI_Crude_Oil':            'The spot price of West Texas Intermediate crude oil, a global benchmark for petroleum pricing. It acts as a primary barometer for global energy demand, inflation pressures, and the health of the industrial economy.',
    '10-Year_Breakeven_Inflation_Rate': 'The market expectation of average inflation over the next 10 years, calculated as the yield difference between nominal 10-Year Treasuries and Treasury Inflation-Protected Securities.',
    'USD_to_JPY':               'The foreign exchange rate representing how many Japanese Yen are needed to purchase one U.S. Dollar. It is a major indicator of trade dynamics between the US and Japan and acts as a proxy for global risk appetite.'
}

topics_dict.update(macro_topics)

# --- Final Prep for OpenAI ---
topic_names = list(topics_dict.keys())

# Format the strings to give the Embedding model explicit context
topic_documents = [f"Topic: {k}. Core concepts: {v}" for k, v in topics_dict.items()]
print(f"Successfully processed {len(topic_names)} topics.")

Loading topic matrix from C:\Users\jonat\Lasso_paper\Narratives\narratives_construction\_data\word_probabilities.csv...
Successfully processed 190 topics.

Sample Topic Document:
Topic: Natural_disasters. Core concepts: water area damage people storm fire flood river mile home coast disaster island fish hurricane official boat weather emergency local


In [ ]:
import h5py

print("--- Validating Topic Alignment ---")

# 1. Load the exact topic strings from the tensor
with h5py.File(H5_PATH, 'r') as f:
    tensor_topics = f['topics'][:].astype(str)

# 2. Convert to sets for easy comparison
tensor_topic_set = set(tensor_topics)
loaded_topic_set = set(topic_names) # topic_names comes from the previous cell

# 3. Calculate matches and mismatches
matched_topics = tensor_topic_set.intersection(loaded_topic_set)
missing_from_loaded = tensor_topic_set - loaded_topic_set
extra_in_loaded = loaded_topic_set - tensor_topic_set

# 4. Print Summary
print(f"Total topics in betas.h5        : {len(tensor_topic_set)}")
print(f"Total loaded topics (CSV+Macro) : {len(loaded_topic_set)}")
print(f"Perfect matches                 : {len(matched_topics)}")

--- Validating Topic Alignment ---
Total topics in betas.h5        : 190
Total loaded topics (CSV+Macro) : 190
Perfect matches                 : 190

✅ SUCCESS: 100% of the topics in your tensor have a matching definition ready for embedding!


In [29]:
def get_embeddings_in_batches(texts: list, batch_size: int = 100, dimensions: int = None) -> list:
    """Fetch embeddings from OpenAI in batches, optionally reducing dimensionality."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding Batches"):
        batch_texts = texts[i:i + batch_size]
        try:
            # Build the API arguments dynamically
            api_args = {
                "input": batch_texts,
                "model": EMBED_MODEL
            }
            # Only pass dimensions if specified (works for text-embedding-3 models)
            if dimensions is not None:
                api_args["dimensions"] = dimensions
                
            response = client.embeddings.create(**api_args)
            
            batch_embeddings = [data.embedding for data in sorted(response.data, key=lambda x: x.index)]
            all_embeddings.extend(batch_embeddings)
        except Exception as e:
            print(f"Error on batch: {e}")
            time.sleep(5) 
    return all_embeddings

# --- Generate OpenAI Embeddings (Smaller Dimensions) ---
# Let's reduce the dimensions from the default 1536 down to 256
TARGET_DIMENSIONS = 256

print(f"Generating Topic Embeddings ({TARGET_DIMENSIONS} dimensions)...")
topic_embeddings = get_embeddings_in_batches(topic_documents, batch_size=100, dimensions=TARGET_DIMENSIONS)
topic_matrix = np.array(topic_embeddings)

print(f"\nGenerating Firm Summary Embeddings ({TARGET_DIMENSIONS} dimensions)...")
firm_embeddings = get_embeddings_in_batches(firm_documents, batch_size=100, dimensions=TARGET_DIMENSIONS)
firm_matrix = np.array(firm_embeddings)

# --- Compute Cosine Similarity ---
print("\nComputing Cosine Similarity Matrix...")
sim_matrix = cosine_similarity(firm_matrix, topic_matrix)

sim_df = pd.DataFrame(sim_matrix, index=permnos, columns=topic_names)

out_path = f"llm_openai_similarity_matrix_{TARGET_DIMENSIONS}d.csv"
sim_df.to_csv(out_path)
print(f"Saved semantic similarity matrix to {out_path}")

Generating Topic Embeddings (256 dimensions)...


Embedding Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Generating Firm Summary Embeddings (256 dimensions)...


Embedding Batches:   0%|          | 0/13 [00:00<?, ?it/s]


Computing Cosine Similarity Matrix...
Saved semantic similarity matrix to llm_openai_similarity_matrix_256d.csv


In [30]:
print("Loading tensor to calculate LASSO selection rates...")

panel_rows = []

with h5py.File(H5_PATH, 'r') as f:
    all_stocks = f['stocks'][:].astype(str)
    all_topics = f['topics'][:].astype(str)
    
    stock_idx_map = {p: i for i, p in enumerate(all_stocks)}
    topic_idx_map = {t: i for i, t in enumerate(all_topics)}
    
    valid_topics = [t for t in topic_names if t in topic_idx_map]
    valid_permnos = [p for p in permnos if p in stock_idx_map]

    for p in tqdm(valid_permnos, desc="Building Panel Dataset"):
        s_idx = stock_idx_map[p]
        stock_betas = f['betas'][s_idx, :, :]
        
        valid_mask = np.isfinite(stock_betas).any(axis=0)
        stock_betas_valid = stock_betas[:, valid_mask]
        
        if stock_betas_valid.shape[1] == 0:
            continue
            
        sel_rates = (stock_betas_valid != 0).mean(axis=1)
        p_idx_sim = permnos.index(p)
        
        for t in valid_topics:
            t_idx_h5 = topic_idx_map[t]
            t_idx_sim = topic_names.index(t)
            
            rate = sel_rates[t_idx_h5]
            sim = sim_matrix[p_idx_sim, t_idx_sim]
            
            panel_rows.append((p, t, rate, sim))

panel = pd.DataFrame(panel_rows, columns=['permno', 'topic', 'sel_rate', 'sim'])

# --- Two-Way Fixed Effects Transformation ---
print("Applying Firm and Topic Fixed Effects...")
def within(df, col):
    ms = df.groupby('permno')[col].transform('mean')
    mt = df.groupby('topic')[col].transform('mean')
    mg = df[col].mean()
    return df[col] - ms - mt + mg

panel['y_w'] = within(panel, 'sel_rate')
panel['x_w'] = within(panel, 'sim')

# --- OLS Regression ---
X = panel['x_w'].values
y = panel['y_w'].values

beta_fe = np.cov(X, y)[0, 1] / np.var(X)
resid = y - beta_fe * X
n_obs = len(y)
se = np.sqrt((resid**2).sum() / (n_obs - 2) / (np.var(X) * n_obs))
t_fe = beta_fe / se
p_fe = 2 * stats.t.sf(abs(t_fe), df=n_obs - 2)

print(f"\n{'='*60}")
print("PANEL OLS RESULTS (LLM Narrative Alignment Test)")
print(f"{'='*60}")
print(f"Coefficient (β)      : {beta_fe:.6f}")
print(f"t-statistic          : {t_fe:.3f}")
print(f"p-value              : {p_fe:.4e}")
print(f"{'='*60}")

Loading tensor to calculate LASSO selection rates...


Building Panel Dataset:   0%|          | 0/1296 [00:00<?, ?it/s]

Applying Firm and Topic Fixed Effects...

PANEL OLS RESULTS (LLM Narrative Alignment Test)
Coefficient (β)      : -0.002463
t-statistic          : -2.048
p-value              : 4.0578e-02


In [32]:
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

print("Setting up Panel Data...")

# 1. Create a numeric ID for topics to satisfy linearmodels' strict index rules
panel['topic_id'] = panel['topic'].astype('category').cat.codes

# 2. Set the MultiIndex using the new numeric topic_id instead of the string
panel_reg = panel.set_index(['permno', 'topic_id'])

# Add a constant (PanelOLS requires this to calculate the intercept properly)
panel_reg['const'] = 1

print("Fitting Two-Way Fixed Effects Panel OLS...")
# entity_effects=True absorbs 'permno'
# time_effects=True absorbs 'topic_id'
model = PanelOLS(
    dependent=panel_reg['sel_rate'], 
    exog=panel_reg[['const', 'sim']], 
    entity_effects=True, 
    time_effects=True
)

# Fit the model and CLUSTER the standard errors by firm
# This is the academic standard to prevent inflated t-statistics
results = model.fit(cov_type='clustered', cluster_entity=True)

print(f"\n{'='*60}")
print("PANEL OLS RESULTS (Narrative Alignment Test)")
print(f"{'='*60}")
print(results.summary.tables[1]) # Prints just the coefficient table
print(f"{'='*60}")

# Extract the exact beta and p-value
beta_fe = results.params['sim']
p_fe = results.pvalues['sim']

Setting up Panel Data...
Fitting Two-Way Fixed Effects Panel OLS...

PANEL OLS RESULTS (Narrative Alignment Test)
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          0.0200     0.0002     88.248     0.0000      0.0195      0.0204
sim           -0.0025     0.0012    -2.0824     0.0373     -0.0048     -0.0001


In [24]:
import os
import json
import concurrent.futures
from tqdm.auto import tqdm

# --- Configuration for Risk Summaries ---
# We use a new cache file so we don't overwrite your "core business" summaries
RISK_CACHE_FILE = os.path.join(CACHE_DIR, 'firm_risk_summaries.json')

def load_risk_summaries():
    if os.path.exists(RISK_CACHE_FILE):
        with open(RISK_CACHE_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_risk_summaries(summary_dict):
    with open(RISK_CACHE_FILE, 'w') as f:
        json.dump(summary_dict, f, indent=4)

def generate_risk_summary_parallel(row):
    permno = str(row['PERMNO'])
    comnam = str(row['COMNAM'])
    
    prompt = (
        f"Describe the primary macroeconomic sensitivities, input costs, and financial risks of the company "
        f"'{comnam}'. Do not describe what they sell; describe what external factors impact their bottom line. "
        f"Maximum 280 characters."
    )
    
    try:
        response = client.chat.completions.create(
            model=GEN_MODEL,
            messages=[
                {"role": "system", "content": "You are a concise financial risk analyst."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=100,
            temperature=0.0 # Deterministic
        )
        return permno, response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error generating for {comnam}: {e}")
        return permno, None

# --- Execute LLM Generation ---
firm_risk_summaries = load_risk_summaries()
missing_risk_firms = firm_df[~firm_df['PERMNO'].isin(firm_risk_summaries.keys())]

if not missing_risk_firms.empty:
    print(f"Generating Risk LLM summaries for {len(missing_risk_firms)} missing firms in parallel...")
    
    MAX_WORKERS = 10
    BATCH_SIZE = 50
    tasks = [row for _, row in missing_risk_firms.iterrows()]
    
    for i in range(0, len(tasks), BATCH_SIZE):
        batch = tasks[i:i + BATCH_SIZE]
        batch_num = (i // BATCH_SIZE) + 1
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(generate_risk_summary_parallel, row): row for row in batch}
            
            for future in tqdm(concurrent.futures.as_completed(futures), total=len(batch), desc=f"Batch {batch_num}"):
                permno, summary = future.result()
                if summary:
                    firm_risk_summaries[permno] = summary
                    
        save_risk_summaries(firm_risk_summaries)

print(f"Ready: {len(firm_risk_summaries)} risk summaries loaded.")

Generating Risk LLM summaries for 1296 missing firms in parallel...


Batch 1:   0%|          | 0/50 [00:00<?, ?it/s]

Batch 2:   0%|          | 0/50 [00:00<?, ?it/s]

Batch 3:   0%|          | 0/50 [00:00<?, ?it/s]

Batch 4:   0%|          | 0/50 [00:00<?, ?it/s]

...

Batch 23:   0%|          | 0/50 [00:00<?, ?it/s]

Batch 24:   0%|          | 0/50 [00:00<?, ?it/s]

Batch 25:   0%|          | 0/50 [00:00<?, ?it/s]

Batch 26:   0%|          | 0/46 [00:00<?, ?it/s]

Ready: 1296 risk summaries loaded.


In [25]:
print("Generating Risk Summary Embeddings...")
# Prepare the ordered lists
risk_permnos = list(firm_risk_summaries.keys())
risk_documents = list(firm_risk_summaries.values())

# Embed the new risk descriptions
risk_firm_embeddings = get_embeddings_in_batches(risk_documents, batch_size=100)
risk_firm_matrix = np.array(risk_firm_embeddings)

print("\nComputing Cosine Similarity Matrix (Risk vs Topics)...")
# We reuse the topic_matrix generated in previous cells
sim_matrix_risk = cosine_similarity(risk_firm_matrix, topic_matrix)

sim_df_risk = pd.DataFrame(sim_matrix_risk, index=risk_permnos, columns=topic_names)

out_path_risk = "llm_risk_similarity_matrix.csv"
sim_df_risk.to_csv(out_path_risk)
print(f"Saved risk semantic similarity matrix to {out_path_risk}")

Generating Risk Summary Embeddings...


Embedding Batches:   0%|          | 0/13 [00:00<?, ?it/s]


Computing Cosine Similarity Matrix (Risk vs Topics)...
Saved risk semantic similarity matrix to llm_risk_similarity_matrix.csv


In [26]:
import scipy.stats as stats

print("Building Panel Dataset for Risk Narratives...")

panel_rows_risk = []

with h5py.File(H5_PATH, 'r') as f:
    all_stocks = f['stocks'][:].astype(str)
    all_topics = f['topics'][:].astype(str)
    
    stock_idx_map = {p: i for i, p in enumerate(all_stocks)}
    topic_idx_map = {t: i for i, t in enumerate(all_topics)}
    
    valid_topics = [t for t in topic_names if t in topic_idx_map]
    valid_permnos = [p for p in risk_permnos if p in stock_idx_map]

    for p in tqdm(valid_permnos, desc="Building Risk Panel"):
        s_idx = stock_idx_map[p]
        stock_betas = f['betas'][s_idx, :, :]
        
        valid_mask = np.isfinite(stock_betas).any(axis=0)
        stock_betas_valid = stock_betas[:, valid_mask]
        
        if stock_betas_valid.shape[1] == 0:
            continue
            
        sel_rates = (stock_betas_valid != 0).mean(axis=1)
        p_idx_sim = risk_permnos.index(p)
        
        for t in valid_topics:
            t_idx_h5 = topic_idx_map[t]
            t_idx_sim = topic_names.index(t)
            
            rate = sel_rates[t_idx_h5]
            sim = sim_matrix_risk[p_idx_sim, t_idx_sim]
            
            panel_rows_risk.append((p, t, rate, sim))

panel_risk = pd.DataFrame(panel_rows_risk, columns=['permno', 'topic', 'sel_rate', 'sim'])

# --- Two-Way Fixed Effects Transformation ---
print("Applying Firm and Topic Fixed Effects...")
# Utilizing the 'within' function defined in your earlier regression cell
panel_risk['y_w'] = within(panel_risk, 'sel_rate')
panel_risk['x_w'] = within(panel_risk, 'sim')

# --- OLS Regression ---
X_r = panel_risk['x_w'].values
y_r = panel_risk['y_w'].values

beta_fe_r = np.cov(X_r, y_r)[0, 1] / np.var(X_r)
resid_r = y_r - beta_fe_r * X_r
n_obs_r = len(y_r)
se_r = np.sqrt((resid_r**2).sum() / (n_obs_r - 2) / (np.var(X_r) * n_obs_r))
t_fe_r = beta_fe_r / se_r
p_fe_r = 2 * stats.t.sf(abs(t_fe_r), df=n_obs_r - 2)

print(f"\n{'='*60}")
print("PANEL OLS RESULTS (RISK EXPOSURE Alignment Test)")
print(f"{'='*60}")
print(f"Coefficient (β)      : {beta_fe_r:.6f}")
print(f"t-statistic          : {t_fe_r:.3f}")
print(f"p-value              : {p_fe_r:.4e}")
print(f"{'='*60}")

Building Panel Dataset for Risk Narratives...


Building Risk Panel:   0%|          | 0/1296 [00:00<?, ?it/s]

Applying Firm and Topic Fixed Effects...

PANEL OLS RESULTS (RISK EXPOSURE Alignment Test)
Coefficient (β)      : -0.002352
t-statistic          : -1.305
p-value              : 1.9197e-01


In [27]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Ensure panel_risk is loaded from the previous cell
print("--- TEST 1: RANK-BASED FIXED EFFECTS OLS ---")

# 1. Rank Transformation
# For each firm, rank the 190 topics by their semantic similarity. 
# Rank 1 = Highest similarity. 
panel_risk['sim_rank'] = panel_risk.groupby('permno')['sim'].rank(method='first', ascending=False)

# Create a Dummy Variable: 1 if the topic is in the firm's Top 5 most similar, 0 otherwise
panel_risk['is_top_5'] = (panel_risk['sim_rank'] <= 5).astype(int)

# 2. Apply the exact same Two-Way Fixed Effects logic to the new Ranked variable
panel_risk['y_w'] = within(panel_risk, 'sel_rate')
panel_risk['x_w_rank'] = within(panel_risk, 'is_top_5')

X_rank = panel_risk['x_w_rank'].values
y_r = panel_risk['y_w'].values

# Calculate beta, SE, and p-value for the Ranked OLS
beta_rank = np.cov(X_rank, y_r)[0, 1] / np.var(X_rank)
resid_rank = y_r - beta_rank * X_rank
n_obs_rank = len(y_r)
se_rank = np.sqrt((resid_rank**2).sum() / (n_obs_rank - 2) / (np.var(X_rank) * n_obs_rank))
t_rank = beta_rank / se_rank
p_rank = 2 * stats.t.sf(abs(t_rank), df=n_obs_rank - 2)

print(f"{'='*60}")
print("PANEL OLS RESULTS (Independent Var: Top 5 Semantic Rank)")
print(f"{'='*60}")
print(f"Coefficient (β)      : {beta_rank:.6f}")
print(f"t-statistic          : {t_rank:.3f}")
print(f"p-value              : {p_rank:.4e}")
print(f"{'='*60}")


print("\n--- TEST 2: LOGISTIC REGRESSION ---")
# 1. Binarize the Dependent Variable
# Instead of a continuous rate heavily clustered at 0, we ask: "Did LASSO EVER select this?"
panel_risk['ever_selected'] = (panel_risk['sel_rate'] > 0).astype(int)

# 2. Estimate the Logistic Regression
# Note: In non-linear models like Logit, including 1,000+ Firm Fixed Effects causes the 
# "Incidental Parameters Problem" (and often crashes memory). 
# Therefore, we control for Topic popularity using C(topic), but pool the firms.
print("Fitting Logit model (This may take 15-30 seconds)...")

try:
    # We predict 'ever_selected' using raw similarity, controlling for Topic Fixed Effects
    logit_model = smf.logit("ever_selected ~ sim + C(topic)", data=panel_risk).fit(disp=0)
    
    # Extract the results specifically for the 'sim' variable
    logit_beta = logit_model.params['sim']
    logit_pval = logit_model.pvalues['sim']
    logit_tstat = logit_model.tvalues['sim']

    print(f"{'='*60}")
    print("LOGISTIC REGRESSION RESULTS (Dep Var: Ever Selected = 1)")
    print(f"{'='*60}")
    print(f"Coefficient (Log-Odds): {logit_beta:.6f}")
    print(f"z-statistic           : {logit_tstat:.3f}")
    print(f"p-value               : {logit_pval:.4e}")
    print(f"{'='*60}")
    
except Exception as e:
    print(f"Logit convergence failed: {e}")

--- TEST 1: RANK-BASED FIXED EFFECTS OLS ---
PANEL OLS RESULTS (Independent Var: Top 5 Semantic Rank)
Coefficient (β)      : 0.000114
t-statistic          : 0.256
p-value              : 7.9800e-01

--- TEST 2: LOGISTIC REGRESSION ---
Fitting Logit model (This may take 15-30 seconds)...
LOGISTIC REGRESSION RESULTS (Dep Var: Ever Selected = 1)
Coefficient (Log-Odds): 0.173809
z-statistic           : 1.625
p-value               : 1.0413e-01


In [28]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Ensure your original 'panel' dataframe and 'within' function are in memory
print("--- TEST 1: RANK-BASED FIXED EFFECTS OLS (CORE BUSINESS NARRATIVES) ---")

# 1. Rank Transformation
# For each firm, rank the 190 topics by their core business semantic similarity. 
panel['sim_rank'] = panel.groupby('permno')['sim'].rank(method='first', ascending=False)

# Create Dummy Variable: 1 if the topic is in the firm's Top 5 most similar, 0 otherwise
panel['is_top_5'] = (panel['sim_rank'] <= 5).astype(int)

# 2. Apply Two-Way Fixed Effects logic to the Ranked variable
panel['y_w'] = within(panel, 'sel_rate')
panel['x_w_rank'] = within(panel, 'is_top_5')

X_rank_core = panel['x_w_rank'].values
y_core = panel['y_w'].values

# Calculate beta, SE, and p-value for the Ranked OLS
beta_rank_core = np.cov(X_rank_core, y_core)[0, 1] / np.var(X_rank_core)
resid_rank_core = y_core - beta_rank_core * X_rank_core
n_obs_rank_core = len(y_core)
se_rank_core = np.sqrt((resid_rank_core**2).sum() / (n_obs_rank_core - 2) / (np.var(X_rank_core) * n_obs_rank_core))
t_rank_core = beta_rank_core / se_rank_core
p_rank_core = 2 * stats.t.sf(abs(t_rank_core), df=n_obs_rank_core - 2)

print(f"{'='*60}")
print("PANEL OLS RESULTS (Independent Var: Top 5 Semantic Rank)")
print(f"{'='*60}")
print(f"Coefficient (β)      : {beta_rank_core:.6f}")
print(f"t-statistic          : {t_rank_core:.3f}")
print(f"p-value              : {p_rank_core:.4e}")
print(f"{'='*60}")


print("\n--- TEST 2: LOGISTIC REGRESSION (CORE BUSINESS NARRATIVES) ---")
# 1. Binarize the Dependent Variable
panel['ever_selected'] = (panel['sel_rate'] > 0).astype(int)

print("Fitting Logit model (This may take 15-30 seconds)...")

try:
    # Predict 'ever_selected' using raw similarity, controlling for Topic Fixed Effects
    logit_model_core = smf.logit("ever_selected ~ sim + C(topic)", data=panel).fit(disp=0)
    
    # Extract the results specifically for the 'sim' variable
    logit_beta_core = logit_model_core.params['sim']
    logit_pval_core = logit_model_core.pvalues['sim']
    logit_tstat_core = logit_model_core.tvalues['sim']

    print(f"{'='*60}")
    print("LOGISTIC REGRESSION RESULTS (Dep Var: Ever Selected = 1)")
    print(f"{'='*60}")
    print(f"Coefficient (Log-Odds): {logit_beta_core:.6f}")
    print(f"z-statistic           : {logit_tstat_core:.3f}")
    print(f"p-value               : {logit_pval_core:.4e}")
    print(f"{'='*60}")
    
except Exception as e:
    print(f"Logit convergence failed: {e}")

--- TEST 1: RANK-BASED FIXED EFFECTS OLS (CORE BUSINESS NARRATIVES) ---
PANEL OLS RESULTS (Independent Var: Top 5 Semantic Rank)
Coefficient (β)      : 0.000489
t-statistic          : 1.156
p-value              : 2.4772e-01

--- TEST 2: LOGISTIC REGRESSION (CORE BUSINESS NARRATIVES) ---
Fitting Logit model (This may take 15-30 seconds)...
LOGISTIC REGRESSION RESULTS (Dep Var: Ever Selected = 1)
Coefficient (Log-Odds): -1.280675
z-statistic           : -13.410
p-value               : 5.2563e-41
